# Demo 5 --- A GMS store from a markdown document

This notebook builds a knowledge store from a single `.md` file, end to end:

1. `build_rag_store` runs the **GEODE loop**, which reads triples from the document, self-corrects them, and trains a geometric memory (a GMS).
2. We read back the **extracted triples**.
3. We draw the trained graph as **geodesic arcs on the embedding sphere**.
4. We **query** it --- a many-to-many relation and a multi-hop chain.

The domain is the family running example from the KG-embedding book: six people across three generations. It is small enough to check by eye, and it carries the two patterns worth showing --- `parentOf` is **many-to-many** (a parent has several children, a child has several parents), and `grandparent` is a **multi-hop** composition of `parentOf` with itself.

## The source document

The input is ordinary markdown. The regex ingester reads each table row as `entity has_<column> value`, so the column headers (`spouse`, `parent_of`, `sibling_of`) become the relation names. We write the file next to the other demo data and display it.

In [ ]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

FAMILY_MD = '# Family Knowledge Base\n\nA tiny knowledge base: six people across three generations. Ann and Bob are\nmarried and are the parents of Carol and Dave; Carol and Eve are married and are\nthe parents of Frank.\n\n## Marriages\n\n| Person | spouse |\n| --- | --- |\n| Ann | Bob |\n| Carol | Eve |\n\n## Parenthood\n\n| Parent | parent_of |\n| --- | --- |\n| Ann | Carol |\n| Ann | Dave |\n| Bob | Carol |\n| Bob | Dave |\n| Carol | Frank |\n| Eve | Frank |\n\n## Siblings\n\n| Person | sibling_of |\n| --- | --- |\n| Carol | Dave |\n| Dave | Carol |\n'

# Resolve the repo data dir (the dir that already holds the demo data).
root = next((c for c in (Path('.'), Path('..'), Path('../..'),
                         Path('../../code'), Path('../code'))
             if (c / 'data').is_dir()), Path('.'))
DATA = root / 'data'
MD = DATA / 'family_kb.md'
STORE = DATA / 'gms_family_store'

MD.write_text(FAMILY_MD)
print(MD.read_text())

## Build the store with the GEODE loop

`build_rag_store` does everything in one call: it ingests the markdown into triples, runs the GEODE self-correction loop (each pass trains a small GMS, flags triples the geometry finds inconsistent, and integrates the survivors), then trains the production GMS on the final clean graph and saves it to disk.

The geometry is deliberately small (`d_v=d_u=32`) because the graph is small; `cap` admissibility with a low `n_boundary` suits six entities. `ingest_mode='regex'` and `llm=None` keep the run deterministic and offline --- no LLM is used to read the document.

In [ ]:
import torch
from knowlytix.core.config import GeometryConfig, TrainConfig, CapLossConfig
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.geode.rag import build_rag_store
from knowlytix.knowledge.geode.loop import make_default_trainer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = DocGMSConfig(
    store_path=str(STORE),
    ingest_mode='regex',          # deterministic table parser, no LLM
    loss_mode='cap',              # relation-conditioned spherical-cap admissibility
    geometry=GeometryConfig(d_v=32, d_u=32, m=16, d=16),
    cap=CapLossConfig(n_boundary=2),   # few entities -> few boundary negatives
    train=TrainConfig(epochs=250, batch_size=32, neg_samples=4,
                      lr=5e-3, lr_riemannian=2e-3),
)

res = build_rag_store(
    str(MD), config, device=device,
    geode_trainer=make_default_trainer(device, epochs=60),
    llm=None,
)
store = res.store
print(f'converged={res.converged}  iterations={res.iterations}')
print(f'entities={res.n_entities}  relations={len(store.adapter.relation_to_idx)}'
      f'  triples={res.n_triples}')
print(f'GEODE corrections={len(res.corrections)}  '
      f'anchor_violations={len(res.anchor_violations)}')

Six entities, three relations, and a handful of triples. The GEODE loop drops the organizational `in_section` edges the ingester emits (they carry no fact), so what remains is the family graph itself. The clean corpus has no contradictions, so GEODE reports zero corrections.

## The extracted triples

These are the facts the store was trained on --- read straight back from the store, grouped by relation. `has_parent_of` is the many-to-many relation: Ann appears with two children, and Frank appears with two parents.

In [ ]:
from collections import defaultdict

by_rel = defaultdict(list)
for h, r, t in sorted(store.query_triples()):
    by_rel[r].append((h, t))

for r in sorted(by_rel):
    print(r)
    for h, t in by_rel[r]:
        print(f'    {h:6s} -> {t}')

## Visualize: geodesics on the embedding sphere

Training places each entity on a sphere in the learned embedding space. We reduce the entity embeddings to three dimensions with PCA, project onto the unit sphere, and draw each triple as a **geodesic arc** (a great-circle segment) from head to tail, colored by relation. For a rotation-operator embedding the head-to-tail path of a relation is exactly such a geodesic. `kg_sphere.py` sits beside the chapter notebooks; we add that directory to the path and load the store straight from disk.

In [ ]:
import sys
# kg_sphere.py lives in the notebooks/ dir; find it walking up from here.
for up in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
    if (up / 'kg_sphere.py').exists():
        sys.path.insert(0, str(up))
        break
import kg_sphere as K

kg = K.load_gms_store(str(STORE), source='v')   # entity embeddings + triples
print(f'{len(kg.labels)} entities, {len(kg.triples)} triples, '
      f'relations={kg.relations}')
K.visualize(kg, show=False)

## Query 1 --- a many-to-many relation

`query_triples` pattern-matches the graph: fix the head to read a person's children, fix the tail to read a person's parents. `parent_of` is many-to-many, so both directions return more than one answer.

In [ ]:
def children_of(name):
    return [t for h, r, t in store.query_triples(head=name, relation='has_parent_of')]

def parents_of(name):
    return [h for h, r, t in store.query_triples(relation='has_parent_of', tail=name)]

print('children of ann :', children_of('ann'))    # one parent, several children
print('parents of carol:', parents_of('carol'))   # one child, several parents
print('parents of frank:', parents_of('frank'))
print('spouse of ann   :', [t for h, r, t in
                            store.query_triples(head='ann', relation='has_spouse')])

## Query 2 --- a multi-hop chain

`grandparent` is not a stored relation. It is `parent_of` composed with `parent_of`: the grandparents of Frank are the parents of Frank's parents. We answer it by walking two hops over the graph.

In [ ]:
def grandparents_of(name):
    out = set()
    for p in parents_of(name):
        out.update(parents_of(p))
    return sorted(out)

print('parents of frank      :', parents_of('frank'))
print('grandparents of frank :', grandparents_of('frank'))

## Calibrate the decision gates

The geometry scores a fact by its geodesic distance to the relation's cap center (lower is more plausible), but a decision needs an operating point, and no gate should read a hardcoded default. `GMSJudge.calibrate` fits one from the store's own graph: for each channel it takes the real triples as positives and corrupted ones as negatives and fits the cut. It calibrates every channel --- geodesic plausibility, two-hop path transport, u-space tension (contradiction), and holonomy (path consistency) --- and reports the accuracy of each.

Calibration also decides *whether a channel is usable at all*. On this graph the geodesic channel separates cleanly, but the tension channel comes back `degenerate`: a family declares no oppositions (`opposite_of`) and no functional relations, so there is nothing for a contradiction cut to fit, and the gate abstains rather than guess. We persist the operating point next to the store.

In [ ]:
import json
from knowlytix.harness.testing.judge import GMSJudge
from knowlytix.harness.governance.reasoner import (
    CalibratedThresholds, GeometricReasoner)

judge = GMSJudge(store)
judge.calibrate()                       # prints the per-channel calibration table
thresholds = CalibratedThresholds.from_judge(judge)
reasoner = GeometricReasoner(store, thresholds, device=device)

# Persist the calibrated operating point (no gate reads a default).
(STORE / 'threshold_calibration.json').write_text(
    json.dumps(judge._thresholds, indent=2, default=str))
print('\naccept threshold (geodesic) =', round(thresholds.tau_plausibility, 3))
print('tension channel            =', thresholds.tension_status)

## A query that is not plausible

`is_plausible` scores a triple and compares it to the calibrated accept threshold. A true edge passes; a fact the graph does not support is rejected. Asking whether Ann is the *parent* of Frank fails --- she is his grandparent, two `parent_of` hops away, so the direct edge lands outside the cap.

In [ ]:
def check(h, rel, t):
    ok, d = reasoner.is_plausible(h, rel, t)
    verdict = 'PLAUSIBLE' if ok else 'rejected'
    print(f'  {h:5s} -{rel[4:]:9s}-> {t:5s}  distance={d:.3f}  {verdict}')

print(f'accept threshold = {thresholds.tau_plausibility:.3f}\n')
check('ann', 'has_parent_of', 'carol')   # true edge
check('ann', 'has_parent_of', 'frank')   # grandparent asked as parent -> rejected

## A contradiction: flip parent and child

`parent_of` is antisymmetric: if Ann is Carol's parent, Carol is not Ann's parent. So flipping the head and tail of a true edge produces a contradiction, and the calibrated gate must reject the flipped triple while accepting the original. We take Ann's children (read above) and check both directions.

In [ ]:
for child in children_of('ann'):
    ok_true, d_true = reasoner.is_plausible('ann', 'has_parent_of', child)
    ok_flip, d_flip = reasoner.is_plausible(child, 'has_parent_of', 'ann')
    print(f'  ann parent_of {child:5s}: {"ok" if ok_true else "no":3s} (d={d_true:.3f})'
          f'   |   flip {child} parent_of ann: '
          f'{"ok" if ok_flip else "CONTRADICTION"} (d={d_flip:.3f})')
    assert ok_true and not ok_flip

# The dedicated tension channel is degenerate here (no declared opposition),
# so this antisymmetry contradiction is caught by the calibrated plausibility
# gate: the flipped edge falls outside the relation's cap.
print('\ntension_status:', thresholds.tension_status,
      '-> contradiction caught by the plausibility gate, not u-space tension')

## Reload check

The store is just files under `store_path`. A fresh `GMSExpertStore` loads it with no rebuild and answers the same queries.

In [ ]:
from knowlytix.knowledge.store import GMSExpertStore

reloaded = GMSExpertStore(config, device)
assert reloaded.load(), f'no store at {STORE}'
assert sorted(reloaded.query_triples()) == sorted(store.query_triples())
assert sorted(h for h, r, t in
        reloaded.query_triples(relation='has_parent_of', tail='frank')) == ['carol', 'eve']
print('reload OK:', len(reloaded.query_triples()), 'triples')

## Summary

One markdown file became a trained, queryable, calibrated GMS store. The GEODE loop extracted the triples and trained the geometry; the store answers pattern queries (including many-to-many relations) directly and multi-hop questions by traversal. `GMSJudge.calibrate` fit an operating point for every decision channel from the graph itself, so plausibility is a calibrated verdict rather than a hardcoded cut: an unsupported fact (Ann as Frank's *parent*) is rejected, and flipping a true edge (`carol parent_of ann`) is caught as a contradiction. Calibration also reported the tension channel as `degenerate` on this corpus --- no declared opposition to fit --- so that gate abstains rather than guess. The same calls scale to real documents; swap the family tables for a financial report and the pipeline is unchanged.